# Figure 08 — Rainfall context of PCS-selected seismic states

Data preparation and statistical analysis follow `010_rainfall_coupling.py`. The complete a–d figure is editable in one plotting cell.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Paths and analysis settings
HERE = Path.cwd()
if HERE.name != "uilt_paper_figure":
    HERE = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure")
BASE = HERE.parent

RAIN_WORKBOOK = BASE / "03visual" / "rain_coupling" / "gpm_rainfall_analysis" / "GPM_IMERG_Illgraben_rainfall_analysis.xlsx"
RAIN_CONTINUOUS_CSV = BASE / "03visual" / "rain_coupling" / "gpm_rainfall_analysis" / "GPM_IMERG_Illgraben_30min_continuous_roll.csv"
MODE_WORKBOOK = BASE / "03visual" / "compared_different_duration" / "mode_rainfall_relationship" / "mode_rainfall_relationship.xlsx"
PCS_WORKBOOK = BASE / "02clustering" / "result" / "DEC_result" / "precursor_curve_metrics" / "precursor_by_metric" / "pcs_precursor_clusters.xlsx"
PCS_SHEET = "selected_precursor"

OUT_DIR = HERE / "Fig7_rainfall_context"; OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_BASE = OUT_DIR / "figure7_rainfall_context_estimation"

RANDOM_SEED = 20260803; N_PERMUTATIONS = 20_000; N_BOOTSTRAPS = 10_000
FOREST_HOURS = list(range(2, 25, 2))
METRICS = [
    {
        "column": f"event_cum_{hours}h_mm",
        "title": f"Cumulative {hours} h rainfall",
        "ylabel": "Rainfall (mm)",
        "timescale": f"last {hours} h",
        "missing_column": f"event_missing_{hours}h_count",
        "hours": hours,
    }
    for hours in FOREST_HOURS
]
ANTECEDENT_METRICS = [{
    "column": "event_cum_7d_mm",
    "title": "Cumulative 7 d rainfall",
    "ylabel": "Rainfall (mm)",
    "timescale": "last 7 d",
    "missing_column": "event_missing_168h_count",
}]
ALL_METRICS = METRICS + ANTECEDENT_METRICS
PCS_GROUPS = ["PCS-selected", "No PCS state"]
PATTERN_GROUPS = ["Pattern I", "Pattern II"]

In [ ]:
# Data utility functions
def normalize_event(value: object) -> str:
    """Return a stable lower-case event label such as ``event12``."""
    match = re.search(r"(\d+)", str(value))
    if not match:
        raise ValueError(f"Cannot parse event identifier: {value!r}")
    return f"event{int(match.group(1))}"


def event_number(value: object) -> int:
    return int(re.search(r"(\d+)", str(value)).group(1))


def require_columns(frame: pd.DataFrame, columns: list[str], source: Path) -> None:
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"Missing columns in {source.name}: {missing}")


def add_even_hour_rainfall_windows(rainfall: pd.DataFrame) -> pd.DataFrame:
    """Calculate complete 2--24 h rainfall windows from the 30 min record."""
    require_columns(
        rainfall,
        ["rain_record_time_utc"],
        RAIN_WORKBOOK,
    )
    continuous = pd.read_csv(
        RAIN_CONTINUOUS_CSV,
        usecols=["time_utc", "rain_mm_30min", "is_missing_record"],
    )
    continuous["time_utc"] = pd.to_datetime(continuous["time_utc"], utc=True)
    continuous = continuous.sort_values("time_utc").reset_index(drop=True)
    if continuous["time_utc"].duplicated().any():
        raise ValueError("Duplicate timestamps in the continuous rainfall record.")
    time_steps = continuous["time_utc"].diff().dropna()
    if not time_steps.eq(pd.Timedelta(minutes=30)).all():
        raise ValueError(
            "The continuous rainfall table is not a complete 30 min time grid."
        )

    rainfall = rainfall.copy()
    rainfall["rain_record_time_utc"] = pd.to_datetime(
        rainfall["rain_record_time_utc"], utc=True
    )
    record_times = rainfall["rain_record_time_utc"]
    rainfall_values = continuous["rain_mm_30min"].fillna(0.0).astype(float)
    missing_values = (
        continuous["is_missing_record"].fillna(True).astype(bool)
        | continuous["rain_mm_30min"].isna()
    ).astype(int)

    computed = pd.DataFrame({"time_utc": continuous["time_utc"]})
    for hours in FOREST_HOURS:
        records = hours * 2
        value_column = f"event_cum_{hours}h_mm"
        missing_column = f"event_missing_{hours}h_count"
        computed[value_column] = rainfall_values.rolling(
            records,
            min_periods=records,
        ).sum()
        computed[missing_column] = missing_values.rolling(
            records,
            min_periods=records,
        ).sum()

    computed = computed.set_index("time_utc")
    missing_matches = sorted(set(record_times).difference(computed.index))
    if missing_matches:
        raise ValueError(
            "Event rainfall timestamps are absent from the continuous record: "
            f"{missing_matches}"
        )
    aligned = computed.loc[record_times].reset_index(drop=True)
    for hours in FOREST_HOURS:
        value_column = f"event_cum_{hours}h_mm"
        missing_column = f"event_missing_{hours}h_count"
        rainfall[value_column] = aligned[value_column].to_numpy(float)
        rainfall[missing_column] = aligned[missing_column].to_numpy(float)

    # Confirm that the new rolling implementation reproduces every rainfall
    # window already available in the event summary table.
    reported_value_columns = {
        2: "event_cum_2h_mm",
        4: "event_cum_4h_mm",
        6: "event_cum_6h_mm",
        8: "event_cum_8h_mm",
        12: "event_cum_12h_mm",
        24: "event_cum_1d_mm",
    }
    for hours, reported_column in reported_value_columns.items():
        original_column = f"_reported_{reported_column}"
        if original_column not in rainfall.columns:
            continue
        calculated_column = f"event_cum_{hours}h_mm"
        if not np.allclose(
            rainfall[original_column],
            rainfall[calculated_column],
            equal_nan=True,
        ):
            raise ValueError(
                f"Recalculated {hours} h rainfall does not match {reported_column}."
            )
    return rainfall


def load_event_table() -> pd.DataFrame:
    """Build the 44-event cohort from the authoritative main-paper PCS table."""
    rainfall = pd.read_excel(RAIN_WORKBOOK, sheet_name="event_rainfall")
    modes = pd.read_excel(MODE_WORKBOOK, sheet_name="mode_rainfall_event_table")
    pcs = pd.read_excel(PCS_WORKBOOK, sheet_name=PCS_SHEET)

    reported_windows = {
        2: "event_cum_2h_mm",
        4: "event_cum_4h_mm",
        6: "event_cum_6h_mm",
        8: "event_cum_8h_mm",
        12: "event_cum_12h_mm",
        24: "event_cum_1d_mm",
    }
    required_rain = [
        "event",
        "time_offset_min",
        "rain_record_time_utc",
        *reported_windows.values(),
        "event_cum_7d_mm",
        "event_missing_168h_count",
    ]
    require_columns(rainfall, required_rain, RAIN_WORKBOOK)
    require_columns(modes, ["event", "mode_short_name"], MODE_WORKBOOK)
    require_columns(pcs, ["event", "cluster", "selection_status"], PCS_WORKBOOK)

    rainfall = rainfall[required_rain].copy()
    for reported_column in reported_windows.values():
        rainfall[f"_reported_{reported_column}"] = rainfall[reported_column]
    rainfall = add_even_hour_rainfall_windows(rainfall)
    modes = modes[["event", "mode_short_name"]].copy()
    rainfall["event"] = rainfall["event"].map(normalize_event)
    modes["event"] = modes["event"].map(normalize_event)
    pcs["event"] = pcs["event"].map(normalize_event)
    pcs["cluster"] = pd.to_numeric(pcs["cluster"], errors="coerce")
    pcs_selected = pcs[
        pcs["selection_status"].astype(str).str.strip().str.lower().eq("selected")
        & pcs["cluster"].ge(0)
    ][["event", "cluster"]].copy()

    if rainfall["event"].duplicated().any():
        duplicated = rainfall.loc[rainfall["event"].duplicated(), "event"].tolist()
        raise ValueError(f"Duplicated events in rainfall table: {duplicated}")
    if modes["event"].duplicated().any():
        duplicated = modes.loc[modes["event"].duplicated(), "event"].tolist()
        raise ValueError(f"Duplicated events in mode table: {duplicated}")
    if pcs_selected["event"].duplicated().any():
        duplicated = pcs_selected.loc[
            pcs_selected["event"].duplicated(), "event"
        ].tolist()
        raise ValueError(f"Duplicated events in main PCS table: {duplicated}")
    if len(rainfall) != 44:
        raise ValueError(f"Expected 44 rainfall events, found {len(rainfall)}")
    if len(pcs_selected) != 35:
        raise ValueError(
            f"Expected 35 main-paper PCS-selected events, found {len(pcs_selected)}"
        )

    pcs_events = set(pcs_selected["event"])
    mode_events = set(modes["event"])
    if mode_events != pcs_events:
        only_mode = sorted(mode_events - pcs_events, key=event_number)
        only_pcs = sorted(pcs_events - mode_events, key=event_number)
        raise ValueError(
            "Pattern labels are not aligned with the authoritative main-paper "
            f"PCS cohort. Only in pattern table: {only_mode}; only in PCS table: "
            f"{only_pcs}. Regenerate the pattern table before plotting."
        )
    invalid_modes = sorted(
        set(modes["mode_short_name"].dropna()).difference(PATTERN_GROUPS)
    )
    if invalid_modes:
        raise ValueError(f"Unexpected pattern labels: {invalid_modes}")

    data = rainfall.merge(modes, on="event", how="left", validate="one_to_one")
    data["pcs_group"] = np.where(
        data["event"].isin(pcs_events), "PCS-selected", "No PCS state"
    )
    data["main_24h_pcs_cluster"] = data["event"].map(
        pcs_selected.set_index("event")["cluster"].astype(int)
    )
    if data["time_offset_min"].abs().max() > 30:
        raise ValueError(
            "At least one debris-flow time is more than 30 min from the matched "
            "GPM record. Review event-to-rainfall alignment."
        )
    data = data.sort_values("event", key=lambda x: x.map(event_number)).reset_index(drop=True)

    print(
        "Rainfall cohort: "
        f"total={len(data)}, PCS-selected={len(pcs_events)}, "
        f"no PCS state={(data['pcs_group'] == 'No PCS state').sum()}"
    )
    print("Pattern counts:", modes["mode_short_name"].value_counts().to_dict())
    return data

In [ ]:
# Statistical functions
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta; positive values mean the first group is larger."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return float(np.sign(x[:, None] - y[None, :]).mean())


def bootstrap_cliffs_delta(
    x: np.ndarray,
    y: np.ndarray,
    rng: np.random.Generator,
    n_bootstraps: int = N_BOOTSTRAPS,
) -> tuple[float, float, float]:
    observed = cliffs_delta(x, y)
    estimates = np.empty(n_bootstraps, dtype=float)
    for index in range(n_bootstraps):
        xb = rng.choice(x, size=len(x), replace=True)
        yb = rng.choice(y, size=len(y), replace=True)
        estimates[index] = cliffs_delta(xb, yb)
    low, high = np.percentile(estimates, [2.5, 97.5])
    return observed, float(low), float(high)


def permutation_test_cliffs_delta(
    x: np.ndarray,
    y: np.ndarray,
    rng: np.random.Generator,
    n_permutations: int = N_PERMUTATIONS,
) -> tuple[float, float]:
    """Two-sided label-permutation test for Cliff's delta."""
    observed = cliffs_delta(x, y)
    pooled = np.concatenate([x, y])
    n_x = len(x)
    exceedances = 0
    for _ in range(n_permutations):
        permuted = rng.permutation(pooled)
        statistic = cliffs_delta(permuted[:n_x], permuted[n_x:])
        exceedances += abs(statistic) >= abs(observed)
    p_value = (exceedances + 1) / (n_permutations + 1)
    return observed, float(p_value)


def holm_adjust(p_values: list[float]) -> np.ndarray:
    """Holm family-wise error correction without external dependencies."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running_max = 0.0
    m = len(p)
    for rank, index in enumerate(order):
        candidate = min((m - rank) * p[index], 1.0)
        running_max = max(running_max, candidate)
        adjusted[index] = running_max
    return adjusted


def comparison_statistics(
    data: pd.DataFrame,
    group_column: str,
    groups: list[str],
    comparison_name: str,
    seed_offset: int,
    complete_case: bool,
    metrics: list[dict[str, object]] = METRICS,
) -> pd.DataFrame:
    """Calculate event-level effect sizes and corrected P values."""
    records: list[dict[str, object]] = []
    raw_p_values: list[float] = []

    for metric_index, metric in enumerate(metrics):
        column = metric["column"]
        valid = data[column].notna()
        if complete_case:
            valid &= data[metric["missing_column"]].fillna(np.inf).eq(0)
        x = data.loc[valid & data[group_column].eq(groups[0]), column].to_numpy(float)
        y = data.loc[valid & data[group_column].eq(groups[1]), column].to_numpy(float)
        if len(x) == 0 or len(y) == 0:
            raise ValueError(f"No data for {comparison_name}, metric={column}")

        rng_boot = np.random.default_rng(RANDOM_SEED + seed_offset + metric_index)
        rng_perm = np.random.default_rng(RANDOM_SEED + seed_offset + 100 + metric_index)
        delta, delta_low, delta_high = bootstrap_cliffs_delta(x, y, rng_boot)
        _, p_raw = permutation_test_cliffs_delta(x, y, rng_perm)
        median_difference = float(np.median(x) - np.median(y))
        raw_p_values.append(p_raw)
        records.append(
            {
                "comparison": comparison_name,
                "analysis_set": (
                    "complete rainfall windows" if complete_case else "all numeric records"
                ),
                "metric": column,
                "timescale": metric["timescale"],
                "group_1": groups[0],
                "group_2": groups[1],
                "n_1": len(x),
                "n_2": len(y),
                "median_1_mm": float(np.median(x)),
                "median_2_mm": float(np.median(y)),
                "median_difference_mm": median_difference,
                "cliffs_delta": delta,
                "cliffs_delta_ci_low": delta_low,
                "cliffs_delta_ci_high": delta_high,
                "p_permutation_cliffs_delta_raw": p_raw,
                "missing_window_records_excluded": bool(complete_case),
            }
        )

    adjusted = holm_adjust(raw_p_values)
    for record, p_adjusted in zip(records, adjusted):
        record["p_holm"] = float(p_adjusted)
    return pd.DataFrame.from_records(records)

In [ ]:
# Data reading and preprocessing
data = load_event_table()
data.head()

In [ ]:
# Statistical calculations
pcs_statistics = comparison_statistics(
    data=data,
    group_column="pcs_group",
    groups=PCS_GROUPS,
    comparison_name="PCS presence",
    seed_offset=0,
    complete_case=True,
)
pattern_statistics = comparison_statistics(
    data=data.loc[data["mode_short_name"].notna()].copy(),
    group_column="mode_short_name",
    groups=PATTERN_GROUPS,
    comparison_name="Pattern I vs Pattern II",
    seed_offset=1_000,
    complete_case=True,
)
statistics = pd.concat([pcs_statistics, pattern_statistics], ignore_index=True)

pcs_sensitivity = comparison_statistics(
    data=data,
    group_column="pcs_group",
    groups=PCS_GROUPS,
    comparison_name="PCS presence",
    seed_offset=2_000,
    complete_case=False,
)
pattern_sensitivity = comparison_statistics(
    data=data.loc[data["mode_short_name"].notna()].copy(),
    group_column="mode_short_name",
    groups=PATTERN_GROUPS,
    comparison_name="Pattern I vs Pattern II",
    seed_offset=3_000,
    complete_case=False,
)
sensitivity_statistics = pd.concat([pcs_sensitivity, pattern_sensitivity], ignore_index=True)

pcs_antecedent = comparison_statistics(
    data=data,
    group_column="pcs_group",
    groups=PCS_GROUPS,
    comparison_name="PCS presence",
    seed_offset=4_000,
    complete_case=True,
    metrics=ANTECEDENT_METRICS,
)
pattern_antecedent = comparison_statistics(
    data=data.loc[data["mode_short_name"].notna()].copy(),
    group_column="mode_short_name",
    groups=PATTERN_GROUPS,
    comparison_name="Pattern I vs Pattern II",
    seed_offset=5_000,
    complete_case=True,
    metrics=ANTECEDENT_METRICS,
)
antecedent_statistics = pd.concat([pcs_antecedent, pattern_antecedent], ignore_index=True)

statistics[[
    "comparison", "metric", "n_1", "n_2", "median_1_mm", "median_2_mm",
    "cliffs_delta", "cliffs_delta_ci_low", "cliffs_delta_ci_high", "p_holm",
]]

In [ ]:
# Plot settings
font_add = 3.5
figsize_mm = (183, 170); mm_per_inch = 25.4; png_dpi = 600; tiff_dpi = 600
font_family = "Arial"; base_fs = 7.0 +font_add; title_fs = 7.5; label_fs = 7.0+font_add;
tick_fs = 6.5+font_add; panel_fs = 9.0+font_add ; annotation_fs = 6.2+font_add; note_fs = 5.8+font_add
axis_lw = 0.75; tick_width = 0.65; grid_lw = 0.55; reference_lw = 0.80
mode_colors = {0: "tab:blue", 1: "#E34040"}
pcs_colors = ["#D97732", "#9AA0A6"]; pattern_colors = ["tab:blue", "#E34040"]
dark = "#222222"; mid_gray = "#555555"; line_gray = "#777777"; light_gray = "#B8B8B8"; grid_color = "#E6E6E6"; zero_band_color = "#F2F2F2"; white = "#FFFFFF"
left = 0.085; right = 0.985; bottom = 0.150; top = 0.930; wspace = 0.2; hspace = 0.2; height_ratios = (0.82, 1.40)
box_positions = [0, 1]; box_width = 0.52; box_whis = 1.5; box_alpha = 0.24
median_lw = 1.15; box_lw = 0.80; scatter_size = 16; scatter_alpha = 0.78; scatter_edge_lw = 0.35; jitter_width = 0.13
box_xlim = (-0.55, 1.55); box_ylim_factor = 1.32; annotation_xy = (0.98, 0.97); title_pad = 4
forest_zero_band = (-0.10, 0.10); forest_xlim = (-0.72, 1.02); forest_xticks = [-0.5, 0.0, 0.5, 1.0]
forest_line_lw = 0.75; forest_regular_lw = 0.95; forest_highlight_lw = 1.35
forest_regular_ms = 4.4; forest_highlight_ms = 5.4; forest_cap_size = 2.8; forest_edge_lw = 0.45; star_fs = 7.0; forest_title_pad = 5
panel_a_xy = (-0.13, 1.08); panel_b_xy = (-0.13, 1.08); panel_c_xy = (-0.13, 1.08); panel_d_xy = (-0.13, 1.08)
note_xy = (0.535, 0.025); save_bbox = "tight"; save_facecolor = "white"; save_pad = 0.04; tiff_compression = "tiff_lzw"
# figure_note = ("Points represent debris-flow events; boxes show median and IQR; forest plots show Cliff's δ with bootstrap 95% CI for cumulative rainfall at 2 h increments. Complete rainfall windows only; P values are Holm-adjusted across 12 windows.")

mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [font_family, "Helvetica", "DejaVu Sans"],
    "font.size": base_fs, "axes.titlesize": title_fs, "axes.labelsize": label_fs,
    "xtick.labelsize": tick_fs, "ytick.labelsize": tick_fs, "axes.linewidth": axis_lw,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.width": tick_width, "ytick.major.width": tick_width,
    "legend.frameon": False,
})
plt.ioff()

# Plot
fig_rainfall = plt.figure(figsize=(figsize_mm[0] / mm_per_inch, figsize_mm[1] / mm_per_inch), facecolor=save_facecolor)
grid = fig_rainfall.add_gridspec(2, 2, height_ratios=height_ratios, left=left, right=right, bottom=bottom, top=top, wspace=wspace, hspace=hspace)
ax_a = fig_rainfall.add_subplot(grid[0, 0]); ax_b = fig_rainfall.add_subplot(grid[0, 1])
ax_c = fig_rainfall.add_subplot(grid[1, 0]); ax_d = fig_rainfall.add_subplot(grid[1, 1])

pcs_stats = statistics.loc[statistics["comparison"].eq("PCS presence")]
pattern_stats = statistics.loc[statistics["comparison"].eq("Pattern I vs Pattern II")]
six_hour_metric = next(metric for metric in METRICS if metric["column"] == "event_cum_6h_mm")
pcs_six_hour = pcs_stats.loc[pcs_stats["metric"].eq("event_cum_6h_mm")].iloc[0]
pattern_six_hour = pattern_stats.loc[pattern_stats["metric"].eq("event_cum_6h_mm")].iloc[0]

boxplot_settings = [
    (ax_a, data, "pcs_group", PCS_GROUPS, pcs_colors, pcs_six_hour, RANDOM_SEED),
    (ax_b, data.loc[data["mode_short_name"].notna()].copy(), "mode_short_name", PATTERN_GROUPS, pattern_colors, pattern_six_hour, RANDOM_SEED + 10),
]
for axis, plot_data, group_column, groups, colors, stats_row, jitter_seed in boxplot_settings:
    valid = plot_data[six_hour_metric["column"]].notna() & plot_data[six_hour_metric["missing_column"]].fillna(np.inf).eq(0)
    arrays = [plot_data.loc[valid & plot_data[group_column].eq(group), six_hour_metric["column"]].to_numpy(float) for group in groups]
    box = axis.boxplot(
        arrays, positions=box_positions, widths=box_width, patch_artist=True, showfliers=False, whis=box_whis,
        medianprops={"color": dark, "linewidth": median_lw},
        whiskerprops={"color": mid_gray, "linewidth": box_lw},
        capprops={"color": mid_gray, "linewidth": box_lw},
        boxprops={"color": mid_gray, "linewidth": box_lw},
    )
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color); patch.set_alpha(box_alpha); patch.set_edgecolor(color)

    rng = np.random.default_rng(jitter_seed)
    for position, (values_plot, color) in enumerate(zip(arrays, colors)):
        jitter = rng.uniform(-jitter_width, jitter_width, size=len(values_plot))
        axis.scatter(
            np.full(len(values_plot), position) + jitter, values_plot,
            s=scatter_size, color=color, alpha=scatter_alpha, edgecolor=white,
            linewidth=scatter_edge_lw, zorder=3,
        )

    labels = [f"{group}\n(n={len(values_plot)})" for group, values_plot in zip(groups, arrays)]
    axis.set_xticks(box_positions, labels); axis.set_ylabel(six_hour_metric["ylabel"])
    axis.grid(axis="y", color=grid_color, linewidth=grid_lw, zorder=0); axis.set_axisbelow(True); axis.set_xlim(*box_xlim)
    data_maximum = max(float(np.nanmax(values_plot)) for values_plot in arrays); axis.set_ylim(0, max(data_maximum * box_ylim_factor, 1.0))

    p_holm = float(stats_row["p_holm"]); p_text = "< 0.001" if p_holm < 0.001 else f"= {p_holm:.3f}"
    annotation = (
        f"$\\delta$ = {stats_row['cliffs_delta']:.2f} "
        f"(95% CI {stats_row['cliffs_delta_ci_low']:.2f}, {stats_row['cliffs_delta_ci_high']:.2f})\n"
        f"$P_{{Holm}}$ {p_text}"
    )
    axis.text(*annotation_xy, annotation, transform=axis.transAxes, ha="right", va="top", fontsize=annotation_fs, color="#303030")

forest_settings = [
    (ax_c, pcs_stats, pcs_colors[0], "PCS-selected versus no PCS state"),
    (ax_d, pattern_stats, pattern_colors[0], "Pattern I versus Pattern II"),
]
for axis, forest_statistics, highlight_color, forest_title in forest_settings:
    forest_rows = forest_statistics.set_index("metric").loc[[metric["column"] for metric in METRICS]].reset_index()
    y_positions = np.arange(len(forest_rows), dtype=float)
    axis.axvspan(*forest_zero_band, color=zero_band_color, zorder=0)
    axis.axvline(0, color=line_gray, linestyle="--", linewidth=reference_lw, zorder=1)
    axis.plot(forest_rows["cliffs_delta"], y_positions, color=light_gray, linewidth=forest_line_lw, zorder=1)

    for y_position, row in zip(y_positions, forest_rows.itertuples(index=False)):
        highlight = row.metric == "event_cum_6h_mm"
        point_color = highlight_color if highlight else line_gray
        line_width = forest_highlight_lw if highlight else forest_regular_lw
        marker_size = forest_highlight_ms if highlight else forest_regular_ms
        axis.errorbar(
            row.cliffs_delta, y_position,
            xerr=np.asarray([[row.cliffs_delta - row.cliffs_delta_ci_low], [row.cliffs_delta_ci_high - row.cliffs_delta]]),
            fmt="o", color=point_color, ecolor=point_color, elinewidth=line_width,
            capsize=forest_cap_size, markersize=marker_size, markeredgecolor=white,
            markeredgewidth=forest_edge_lw, zorder=3,
        )
        if float(row.p_holm) < 0.05:
            axis.text(row.cliffs_delta, y_position - 0.28, "*", ha="center", va="bottom", fontsize=star_fs, color=point_color)

    axis.set_yticks(y_positions, [f"{hours} h" for hours in FOREST_HOURS]); axis.invert_yaxis()
    axis.set_xlim(*forest_xlim); axis.set_xticks(forest_xticks); axis.set_xlabel("Cliff's $\\delta$ (first group higher →)")
    axis.grid(axis="x", color=grid_color, linewidth=grid_lw, zorder=0); axis.set_axisbelow(True)

# Axis and layout
for axis, label, panel_xy in [(ax_a, "a", panel_a_xy), (ax_b, "b", panel_b_xy), (ax_c, "c", panel_c_xy), (ax_d, "d", panel_d_xy)]:
    axis.text(*panel_xy, label, transform=axis.transAxes, fontsize=panel_fs, fontweight="bold", va="top", ha="left")

# fig_rainfall.text(*note_xy, figure_note, ha="center", va="bottom", fontsize=note_fs, color=mid_gray)

# Save save_bbox = "tight"
# 全框线，上、右无刻度
for axis in (ax_a,ax_b,ax_c,ax_d):
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(axis_lw)
    axis.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)
    axis.tick_params(axis="both",which="major",length=3,width=tick_width)

# 保存 Fig 8
from pathlib import Path
save_root=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root/"png").mkdir(parents=True,exist_ok=True)
(save_root/"tiff").mkdir(parents=True,exist_ok=True)
png_path=save_root/"png"/"Fig 8.png"
tiff_path=save_root/"tiff"/"Fig 8.tiff"
fig_rainfall.savefig(png_path,format="png",dpi=png_dpi,bbox_inches=save_bbox,facecolor=save_facecolor,pad_inches=save_pad)
fig_rainfall.savefig(tiff_path,format="tiff",dpi=tiff_dpi,bbox_inches=save_bbox,facecolor=save_facecolor,pad_inches=save_pad,pil_kwargs={"compression":tiff_compression})
display(fig_rainfall)
plt.close(fig_rainfall)
print("Saved Fig 8:")
print(png_path)
print(tiff_path)

In [ ]:
# Source-data export
source_columns = [
    "event", "pcs_group", "mode_short_name", "main_24h_pcs_cluster", "time_offset_min",
    *[metric["column"] for metric in ALL_METRICS],
    *sorted({metric["missing_column"] for metric in ALL_METRICS}),
]
source_data = data[source_columns].copy()
source_data.to_csv(OUT_DIR / "figure7_rainfall_context_source_data.csv", index=False)
statistics.to_csv(OUT_DIR / "figure7_rainfall_context_statistics.csv", index=False)
sensitivity_statistics.to_csv(OUT_DIR / "figure7_rainfall_context_sensitivity_all_records.csv", index=False)
antecedent_statistics.to_csv(OUT_DIR / "figure7_rainfall_context_exploratory_7d.csv", index=False)

metadata = pd.DataFrame({
    "item": [
        "statistical unit", "authoritative PCS cohort", "pattern cohort", "primary completeness rule",
        "effect size", "confidence interval", "hypothesis test", "multiple testing",
        "exploratory antecedent context", "interpretation boundary",
    ],
    "definition": [
        "one debris-flow event",
        "35 events selected in the main event-specific 24 h DEC/PCS analysis",
        "Pattern I/II labels validated against the main PCS cohort",
        "exclude an event for a metric when its corresponding rainfall window contains one or more missing 30 min records",
        "Cliff's delta; positive values indicate the first displayed group is higher",
        f"event bootstrap percentile 95% CI ({N_BOOTSTRAPS:,} resamples)",
        f"two-sided label-permutation test of Cliff's delta ({N_PERMUTATIONS:,} permutations)",
        "Holm correction across 12 cumulative-rainfall windows (2--24 h at 2 h increments) per comparison",
        "7 d cumulative rainfall is exported separately and is not shown in the main figure",
        "rainfall association/context, not a deterministic triggering mechanism",
    ],
})
with pd.ExcelWriter(OUT_DIR / "figure7_rainfall_context_source_data.xlsx", engine="openpyxl") as writer:
    metadata.to_excel(writer, sheet_name="metadata", index=False)
    source_data.to_excel(writer, sheet_name="event_data", index=False)
    statistics.to_excel(writer, sheet_name="figure_statistics", index=False)
    sensitivity_statistics.to_excel(writer, sheet_name="sensitivity_all_records", index=False)
    antecedent_statistics.to_excel(writer, sheet_name="exploratory_7d_context", index=False)

print(f"Saved figure and source data to: {OUT_DIR}")